# ModernBERT TSM — Phase 1 Training (Kaggle)

One-shot notebook: clones the repo, regenerates the per-specifier training data,
trains the Phase-1 subset (`in`, `after`, `before`) on `answerai/ModernBERT-base`,
merges them by parameter averaging, and evaluates on `timeqa` / `nobel_prize`.

### Required notebook settings (top-right)
- **Internet: ON** (needed: git clone + HuggingFace model download)
- **Accelerator: GPU** (T4/P100)

### Timing
Defaults = 3 specifiers × 5 epochs. On a T4 this is roughly 1-2 h per specifier.
To **smoke-test** the pipeline first, set `TOTAL_EPOCHS = 1` and
`SPECIFIERS = ["in"]`, then re-run with the real config (training skips
specifiers whose `best_model` already exists).

Results land in `/kaggle/output` (downloadable from the notebook's output tab).


## Configuration
Edit the `CONFIG` dict below before running. Training is resumable: if a
specifier's `best_model` checkpoint already exists it is skipped (`RESUME`).

In [ ]:

import os, sys, subprocess, glob, shutil, json, time

REPO   = "/kaggle/working/TSM"
OUT    = "/kaggle/output"
os.makedirs(OUT, exist_ok=True)

# ============================= CONFIG =============================
BASE_MODEL      = "answerai/ModernBERT-base"
SPECIFIERS      = ["in", "after", "before"]   # Phase-1 subset
TOTAL_EPOCHS    = 5                            # set 1 for a quick smoke test
PER_GPU_BATCH   = 16
NEGATIVE_CTXS   = 5
RESUME          = True     # skip specifiers with an existing best_model
RUN_MERGE       = True
RUN_EVAL        = True
EVAL_DATASETS   = ["timeqa", "nobel_prize"]
REPO_URL        = "https://github.com/AmnO-O/TestingSystem.git"
# ===================================================================

def run(cmd, cwd=None, check=True):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    tail = r.stdout[-3000:] + r.stderr[-6000:]
    print(f"$ {cmd[:180]}\n  <- rc={r.returncode} in {time.time()-t0:.0f}s")
    if check and r.returncode != 0:
        print(tail)
        raise RuntimeError(f"command failed (rc={r.returncode}): {cmd}")
    return r.stdout

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("WARNING: no GPU detected -- finetuning.py will fail on model.cuda()")


In [ ]:

if not os.path.exists(os.path.join(REPO, "tsm.py")):
    run(f"git clone --depth 1 {REPO_URL} {REPO}")
else:
    run(f"cd {REPO} && git pull", check=False)
assert os.path.exists(os.path.join(REPO, "contriever", "finetuning.py")), "repo clone failed"
print("repo ready:", REPO)


In [ ]:

# Kaggle ships torch; make sure transformers is new enough for ModernBERT
# finetuning.py imports src/beir_utils.py -> needs beir (1.x layout).
# Pin beir==1.0.0: beir 2.x is an incompatible rewrite and its module
# paths (beir.retrieval.search.dense, beir.reranking) no longer exist.
run("pip install -q -U transformers tokenizers safetensors beir==1.0.0 faiss-cpu pytrec-eval")

from transformers import ModernBertModel
print("transformers supports ModernBertModel")

# Guard against tokenizer.save_vocabulary being unavailable in newer transformers
utils_path = os.path.join(REPO, "contriever/src/utils.py")
src = open(utils_path, encoding="utf-8").read()
old = """    tokenizer.save_pretrained(epoch_path)
    tokenizer.save_vocabulary(epoch_path)"""
new = """    tokenizer.save_pretrained(epoch_path)
    try:
        tokenizer.save_vocabulary(epoch_path)
    except Exception:
        pass"""
if old in src:
    open(utils_path, "w", encoding="utf-8").write(src.replace(old, new))
    print("patched contriever/src/utils.py (save_vocabulary guard)")


In [ ]:

DATA_DIR = os.path.join(REPO, "dataset/train/TimeQA")
need = [os.path.join(DATA_DIR, f"{split}_{spec}.jsonl")
        for split in ("train", "dev") for spec in SPECIFIERS]
if all(os.path.exists(p) for p in need):
    print("training .jsonl already present, skipping regeneration")
else:
    run("python process_data.py", cwd=DATA_DIR)
found = sorted(os.path.basename(p) for p in glob.glob(os.path.join(DATA_DIR, "*.jsonl")))
print("generated:", found)


## Train per specifier
Each specifier finetunes the base ModernBERT checkpoint with 5 hard negatives
(via `contriever/finetuning.py`) and keeps the best dev-accuracy checkpoint at
`model/modernbert_{spec}/checkpoint/best_model`.

Because the recipe trains by epochs (`TOTAL_EPOCHS>0`), at **every epoch end** finetuning.py logs:
```
[epoch N] train loss: ... | train accuracy: ...   (averaged over that epoch)
eval acc / eval mrr                                (dev set; picks best_model on improvement)
```


In [ ]:

os.environ["TOKENIZERS_PARALLELISM"] = "false"
trained = 0
for spec in SPECIFIERS:
    out_dir  = f"/kaggle/working/model/modernbert_{spec}"
    best     = os.path.join(out_dir, "checkpoint", "best_model")
    train_f  = os.path.join(DATA_DIR, f"train_{spec}.jsonl")
    dev_f    = os.path.join(DATA_DIR, f"dev_{spec}.jsonl")
    if RESUME and os.path.exists(best):
        print(f"[skip] {spec}: best_model already exists"); continue
    print("=" * 72)
    print(f"TRAIN specifier: {spec}  (epochs={TOTAL_EPOCHS}, bs={PER_GPU_BATCH})")
    print("=" * 72)
    run(f"python {REPO}/contriever/finetuning.py "
        f"--model_path {BASE_MODEL} "
        f"--train_data {train_f} --eval_data {dev_f} "
        f"--negative_ctxs {NEGATIVE_CTXS} --negative_hard_ratio 1.0 "
        f"--total_epochs {TOTAL_EPOCHS} --eval_freq 50 --log_freq 10 "
        f"--output_dir {out_dir} "
        f"--per_gpu_batch_size {PER_GPU_BATCH} --per_gpu_eval_batch_size 64")
    assert os.path.exists(best), f"best_model missing for {spec}"
    trained += 1
print(f"finished training {trained} specifier(s)")


In [ ]:

merged = "/kaggle/working/model/merged_model_small"
if not RUN_MERGE:
    print("merge skipped (RUN_MERGE=False)")
else:
    paths = [f"/kaggle/working/model/modernbert_{s}/checkpoint/best_model" for s in SPECIFIERS]
    missing = [p for p in paths if not os.path.exists(p)]
    assert not missing, f"missing checkpoints: {missing}"
    run(f"python {REPO}/train/merge_models.py "
        f"--model_paths {' '.join(paths)} --output_dir {merged} --mode avg")
    assert os.path.exists(os.path.join(merged, "checkpoint.pth"))
    print("merged ok:", merged)


In [ ]:

if not RUN_MERGE:
    merged = "/kaggle/working/model/merged_model_small"
if RUN_EVAL:
    beir_dir = os.path.join(REPO, "dataset/test")
    for ds in EVAL_DATASETS:
        if not os.path.isdir(os.path.join(beir_dir, ds)):
            print(f"[skip eval] {ds}: corpus not present"); continue
        print("=" * 72); print("EVAL", ds); print("=" * 72)
        run(f"python {REPO}/contriever/eval_beir.py "
            f"--model_name_or_path {merged} --beir_dir {beir_dir} --dataset {ds} "
            f"--output_dir {OUT}/{ds} --per_gpu_batch_size 256")
else:
    print("evaluation skipped (RUN_EVAL=False)")


In [ ]:

shutil.copytree(merged, os.path.join(OUT, "merged_model_small"), dirs_exist_ok=True)
with open(os.path.join(OUT, "summary.txt"), "w") as f:
    f.write("SPECIFIERS=%s\nTOTAL_EPOCHS=%s\nBASE_MODEL=%s\n" % (SPECIFIERS, TOTAL_EPOCHS, BASE_MODEL))
    f.write("merged=%s\n" % merged)
print("saved /kaggle/output:")
for p in sorted(glob.glob(OUT + "/**/*", recursive=True)):
    if os.path.isfile(p):
        print("  ", os.path.relpath(p, OUT), f"({os.path.getsize(p)//1024} KB)")
print("DONE. Download your merged model + eval logs from the output tab.")


## Where to find results
- Merged checkpoint: `/kaggle/output/merged_model_small/` (download via the
  notebook's **Output** tab; it persists across versions in Kaggle's output).
- BEIR logs: `/kaggle/output/{dataset}/run.log` and the printed metrics.

To later evaluate the merged model in a fresh notebook (or locally):
```python
python contriever/eval_beir.py --model_name_or_path <merged> --beir_dir dataset/test --dataset timeqa
```
